# Import des librairies

In [453]:
import numpy as np

# Lecture du fichier source

In [454]:
with open ("test.txt") as fichier:
    grille = fichier.readlines()
grille = [line.replace("\n","") for line in grille]
grille

['....#.....',
 '.........#',
 '..........',
 '..#.......',
 '.......#..',
 '..........',
 '.#..^.....',
 '........#.',
 '#.........',
 '......#...']

# Nettoyage du fichier

In [455]:
# On récupère la position (horizontale et verticale) et la direction de l'agent au début
dep_row_agent, dep_col_agent = next((num_line,line.index("^")) for num_line,line in enumerate(grille) if "^" in line)  # next permet de s'atrreter quand on trouve "^"
# On historise les positions de l'agent
dep_direction_agent = "haut" # direction du regard de l'agent

# Maitenant qu'on a identifié la position de l'agent on supprime le "^"
grille = [line.replace("^",".") for line in grille]


# Fonction avancer

In [456]:
# fonction qui détermine si l'agent peut avancer, si oui, on avance, sinon on tourne et on avance
def func_avancer(var_row_agent:int,
                 var_col_agent:int,
                 var_direction_agent:str,
                 var_grille:list[str]
                )->tuple[bool, int, int, str] | None:
    in_grille=True # Par défaut on est dans la grille
    # On définit un dictionnaire qui associe chaque direction à un déplacement dans la grille
    dico_moove={"haut":   (-1, 0),
                "droite": (0, 1),
                "bas":    (1, 0),
                "gauche": (0, -1)
                }
    
    list_directions = ["haut","droite","bas","gauche"]

    limit_v_grille = len(var_grille)-1
    limit_h_grille = len(var_grille[0])-1
    
    # On définit la nouvelle position si on continue dans la même direction (new pos = ancinne position + déplacement)
    new_row_agent = var_row_agent + dico_moove[var_direction_agent][0]  # Déplacement vertical
    new_col_agent = var_col_agent + dico_moove[var_direction_agent][1]  # Déplacement horizontal

    # Si on dépasse de la grille on s'arrete
    if not (0<=new_row_agent<=limit_v_grille and 0<=new_col_agent<=limit_h_grille):
        # print(f"sortie de la grille: {new_row_agent,new_col_agent}")
        in_grille=False
        return (in_grille, var_row_agent, var_col_agent, var_direction_agent)
    
    # Si l'agent ne plus avancer dans la même direction
    if var_grille[new_row_agent][new_col_agent] == "#":
        # On définit la nouvelle direction
        new_direction_agent = list_directions[(list_directions.index(var_direction_agent)+1)%4]

        # On choisit un autre déplacement 
        new_row_agent = var_row_agent + dico_moove[new_direction_agent][0]  # Déplacement vertical
        new_col_agent = var_col_agent + dico_moove[new_direction_agent][1]  # Déplacement horizontal

    
        

    # Si l'agent peut encore avancer dans la même direction
    else:
        # On définit la nouvelle direction (qui reste la même dans ce cas)
        new_direction_agent = var_direction_agent
    # print(f"Ancienne direction : {var_direction_agent}")
    # print(f"Continuer dans la même direction: {continuer}")
    # print(f"Nouvelle direction : {new_direction_agent}")
        
        
    # On relance la fonction    
    return in_grille, new_row_agent, new_col_agent, new_direction_agent        
        

# Fonction tourne en rond

In [457]:
def func_tourne_en_rond(var_grille: list[str],
                        var_row_agent: int,
                        var_col_agent: int,
                        var_direction_agent: str,
                       )->bool:

    historique_deplacements = [(var_row_agent,var_col_agent,var_direction_agent)]
    
    res_func_avancer = ""
    # Tant qu'on ne sort pas de la grille
    while res_func_avancer is not None:
        # print(f"nouvelle position : {pos_row_agent,pos_col_agent}")
        # On avance
        # print(f"pos : {var_row_agent, var_col_agent, var_direction_agent}")
        continuer, var_row_agent, var_col_agent, var_direction_agent = func_avancer(var_row_agent, var_col_agent, var_direction_agent, var_grille)
        # Si on est encore dans la grille
        if continuer is True:
            # print(f"pos2 : {var_row_agent, var_col_agent, var_direction_agent}\n")
            
            pos_agent=(var_row_agent, var_col_agent, var_direction_agent)
            # Si l'agent n'est pas déja passé par cette position
            if pos_agent not in historique_deplacements:
                historique_deplacements.append(pos_agent)
            else:
                # print("Tu tourne en rond chef")
                return True
        else:
            # print("On est sortit de la boucle")
            return False



# On récupere les positions où on peut mettre un obstacle

In [458]:
# On recupere l'emplacement des obtacles deja présents
pos_occupee = [(num_line, index) 
             for num_line, line in enumerate(grille) 
             for index, char in enumerate(line) if char == "#"]
# On ajoute l'emplacement ititial de l'agent 
pos_occupee.append((dep_row_agent,dep_col_agent))
pos_occupee


[(0, 4), (1, 9), (3, 2), (4, 7), (6, 1), (7, 8), (8, 0), (9, 6), (6, 4)]

# On regarde pour chaque position qui n'est pas occupée si en mettant un obstacle, l'agent tourne en rond

In [459]:
# Initialisations
nb_tourne_en_rond = 0
nb_lignes_grille = len(grille)
nb_col_grille = len(grille[0])

# Pour chaque positon
for row_new_obstacle in range(nb_lignes_grille):
    for col_new_obstacle in range(nb_col_grille):
        print(col_new_obstacle)

        # Ajout de l'obtacle
        new_grille = grille[:]
        new_grille[row_new_obstacle] = grille[row_new_obstacle][:col_new_obstacle] + "#" + grille[row_new_obstacle][col_new_obstacle+1:]
        # print(f"position : {(row_new_obstacle,col_new_obstacle,)}")

        # on vérifie qu'elle n'est pas occupée
        if (row_new_obstacle, col_new_obstacle) not in pos_occupee:
            res_tourne_en_rond = func_tourne_en_rond(new_grille, dep_row_agent, dep_col_agent, dep_direction_agent)
            # print(res_tourne_en_rond)
            if res_tourne_en_rond is True :
                nb_tourne_en_rond+=1
    
print(nb_tourne_en_rond)


0
1
2
3
4
5
6
7
8
9
0
1
2
3
4
5


KeyboardInterrupt: 